## 1. Dependency

In [1]:
import csv, re, sys

## 2. Konfigurasi

In [ ]:
dataset = "1sample"

CSV_INPUT = f"dataset/{dataset}.csv"
CSV_OUTPUT = f"results/{dataset}/result-1-normalization.csv"

In [3]:
def count_lines(path):
    # fast line count
    cnt = 0
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for _ in f:
            cnt += 1
    return cnt

## 3. Fungsi Normalisasi URL Encoding

In [4]:
def normalize_url_encoding(url):
    """
    Normalisasi URL encoding pada URL.
    
    1. Convert ~xx to %xx (contoh: ~20 menjadi %20)
    2. Replace + with %20 (treat + as space)
    3. Replace literal spaces with %20
    
    Parameters:
    - url: String URL yang akan dinormalisasi
    
    Returns:
    - String URL yang sudah dinormalisasi
    """
    
    # 1. Convert ~xx to %xx (contoh: ~20, ~3D, ~2F)
    url = re.sub(r'~([0-9A-Fa-f]{2})', r'%\1', url)
    
    # 2. Replace + with %20
    url = url.replace('+', '%20')
    
    # 3. Replace literal spaces with %20
    url = url.replace(' ', '%20')
    
    return url

In [5]:
def normalize_message(message):
    """
    Normalisasi message.
    - Jika http_request: normalisasi hanya bagian URL
    - Jika bukan http_request: kembalikan message apa adanya
    
    Parameters:
    - message: String message lengkap
    
    Returns:
    - String message yang sudah dinormalisasi
    """
    
    # Jika bukan http_request, kembalikan apa adanya
    if not message.startswith('http_request:'):
        return message
    
    # Pattern untuk menangkap: http_request: METHOD URL HTTP/x.x
    # Group 1: bagian awal (http_request: METHOD )
    # Group 2: URL path
    # Group 3: bagian akhir (HTTP/x.x ...)
    pattern = r'^(http_request:\s*(?:GET|POST|PUT|DELETE|PATCH|HEAD|OPTIONS)\s+)(\S+)(\s+HTTP/.*)$'
    
    match = re.match(pattern, message)
    
    if match:
        prefix = match.group(1)  # http_request: GET 
        url = match.group(2)      # /wp-content/...
        suffix = match.group(3)   # HTTP/1.1 from: ...
        
        # Normalisasi hanya bagian URL
        normalized_url = normalize_url_encoding(url)
        
        return prefix + normalized_url + suffix
    
    # Jika tidak match pattern, kembalikan message asli
    return message

## 4. Eksekusi Normalisasi

In [6]:
total_lines = count_lines(CSV_INPUT)
print(f"Total lines in {CSV_INPUT}: {total_lines}")

processed = 0

with open(CSV_INPUT, newline='', encoding="utf-8", errors='replace') as f, open(CSV_OUTPUT, "w", newline='', encoding="utf-8") as out:
    reader = csv.DictReader(f)
    writer = csv.writer(out)
    
    # Tambahkan kolom baru di awal
    fieldnames = ["event_id"] + reader.fieldnames + ["normalized"]
    writer = csv.DictWriter(out, fieldnames=fieldnames)
    writer.writeheader()
    
    for row in reader:
        processed += 1

        # Ambil dan normalisasi kolom message
        original_message = row["message"]
        normalized_message = normalize_message(original_message)

        # Replace message
        row["normalized"] = normalized_message
        row["event_id"] = processed

        writer.writerow(row)
            
        # Progress tiap 50.000 baris
        if processed % 50000 == 0:
            print(f"Processed {processed}/{total_lines} lines ({processed/total_lines:.2%})")

print(f"Conversion finished: {processed}/{total_lines} lines processed.")


Total lines in dataset/organization-x.csv: 216755
Processed 50000/216755 lines (23.07%)
Processed 100000/216755 lines (46.14%)
Processed 150000/216755 lines (69.20%)
Processed 200000/216755 lines (92.27%)
Conversion finished: 216754/216755 lines processed.
